# 도메인 평가 에이전트 — 데이터센터 관점

KV cache 최적화 기술 두 건이 **데이터센터** 환경에서 각각 어떤 조건에서 적합하다고
평가받는지 웹 검색 기반 RAG로 근거를 모아 판단합니다.

- SW: DeepSeek-V2 MLA — 저차원 잠재 압축으로 KV cache 93.3% 감소
- HW: ITME — CXL-Hybrid 계층 메모리로 추론 처리량 1.80배 향상

우열을 판정하지 않고, 관점에 따라 평가가 어떻게 갈리는지를 근거와 함께 기록합니다.

```
질문 생성(한/영) → 웹 검색(출처 등급 필터) → 본문 수집
                      ├ 짧은 문서: 그대로 근거
                      └ 긴 문서: 청킹 → BM25+dense 색인 → 해당 부분만 근거
  → 적용 조건 검토 → 근거 충분성 판단 ─(부족)→ 질의 보완 → 재검색
                                      └(충분)→ 적합성 분석
  → guard(결정적) → linter(표현) → judge(coverage·neutrality)
```

## 왜 데이터센터이고, 왜 임베딩인가

**데이터센터 고정.** KV cache는 연산 병목을 메모리 병목으로 바꾼 문제이고, 그 압박이
실제 비용이 되는 곳이 데이터센터입니다. 온디바이스에서는 한 사용자가 자기 메모리를 쓰고
끝나지만, 데이터센터에서는 한 요청의 KV cache가 다른 사용자 몫 HBM을 잠식해 동시 처리
요청 수를 떨어뜨립니다. 또 ITME가 전제하는 CXL은 랙 스케일 규격이라 온디바이스에 없어서,
SW 압축과 HW 확장이 같은 무대에서 비교되는 환경은 데이터센터뿐입니다.

**임베딩이 필요한 이유는 측정으로 확인했습니다.** 이 에이전트는 평가 질문을 한국어로
만드는데 근거 문서는 영어입니다. 키워드 검색(BM25)만으로 되는지 재봤습니다.

| 방식 | 한국어 질의 Hit@5 | 영어 질의 Hit@5 |
|---|---|---|
| BM25 | 0.29 (2/7) | 1.00 (9/9) |
| dense (bge-m3) | 0.86 (6/7) | 0.89 (8/9) |

BM25는 한국어 질의에서 무너집니다. 겹치는 어휘가 없으니 당연합니다. 교차언어 검색을
성립시키는 것이 임베딩이고, 동시에 영어 질의에서는 BM25가 더 나으므로 둘 다 씁니다.
자세한 근거는 `docs/DOMAIN_AGENT.md`, 재현은 `python -m agents.domain.evaluation.ablation` 참조.

In [1]:
# 노트북이 notebooks/ 안에 있으므로 레포 루트를 import 경로에 넣는다.
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env", override=True)

# TAVILY_API_KEY 가 없어도 동작한다. 그 경우 캐시를 재생하는 오프라인 목 모드로 떨어진다.
import os

assert os.environ.get("OPENAI_API_KEY"), ".env 에 OPENAI_API_KEY 가 필요합니다"
print("루트:", ROOT)
print("웹 검색 모드:", "Tavily" if os.environ.get("TAVILY_API_KEY") else "오프라인 목(캐시 재생)")

루트: /Users/sanlee/Desktop/SKALA/코딩파일/Ai-service/Capstone_Ai_RAG
웹 검색 모드: Tavily


## 1. 출처 정책

검색 제공자의 도메인 필터는 신뢰하지 않습니다. Tavily `include_domains` 에 13개를
지정했더니 medium·substack·youtube 가 그대로 반환되는 것을 확인했습니다(2개일 때는 정상).
그래서 결과를 받은 뒤 등급표로 다시 거르고, 거른 이유를 검색 로그에 남깁니다.

In [2]:
from agents.domain.rag.websearch import SOURCE_TIERS, classify_source

# 등급이 곧 근거 강도다. 매체 보도만 근거인 주장은 direct_evidence 로 올리지 않는다.
for tier, domains in SOURCE_TIERS.items():
    print(f"{tier:<10} {len(domains):>2}개  예: {', '.join(domains[:3])}")

print("\n등급 판정 예시")
for url in [
    "https://arxiv.org/abs/2405.04434",
    "https://www.nvidia.com/en-us/data-center/",
    "https://computeexpresslink.org/spec",
    "https://medium.com/@someone/kv-cache",
]:
    verdict = classify_source(url) or "거부"
    print(f"  {verdict:<14} {url}")

paper       9개  예: arxiv.org, ieee.org, acm.org
patent      3개  예: patents.google.com, patentscope.wipo.int, uspto.gov
standard    4개  예: computeexpresslink.org, jedec.org, opencompute.org
vendor     13개  예: nvidia.com, intel.com, amd.com
news       10개  예: reuters.com, bloomberg.com, theregister.com

등급 판정 예시
  paper          https://arxiv.org/abs/2405.04434
  vendor         https://www.nvidia.com/en-us/data-center/
  standard       https://computeexpresslink.org/spec
  거부             https://medium.com/@someone/kv-cache


In [3]:
# 검색 제공자를 만든다. 질의 단위 파일 캐시를 쓰는 이유는 재현성이다.
# 같은 질의가 실행마다 다른 결과를 주면 ablation 비교도 보고서 재생성도 성립하지 않는다.
from agents.domain.rag.websearch import build_search_provider

search_provider = build_search_provider(ROOT / "data/search_cache")
print("제공자:", type(search_provider).__name__)

제공자: TavilyWebSearch


## 2. 입력 State

도메인 노드는 `request` 와 선행 단계인 기술 조사 결과만 봅니다.
시장·이해관계자 관점의 중간 결론을 같이 넘기면 도메인 판단이 그쪽 결론에 물들기 때문에,
`project_input()` 이 입력을 추려서 서브그래프에 넘깁니다.
아래에 시장·이해관계자 결과를 일부러 넣어 두고, 마지막에 누수 여부를 확인합니다.

In [4]:
state = {
    "run_id": "domain-nb-001",
    "request": {
        "sw": {
            "name": "DeepSeek-V2 MLA (Multi-head Latent Attention)",
            "selection_reason": "저차원 잠재 압축으로 KV cache 93.3% 감소",
            "seed_urls": [],
        },
        "hw": {
            "name": "ITME (Inference Tiered Memory Expansion, CXL-Hybrid)",
            "selection_reason": "CXL-Hybrid 계층 메모리로 추론 처리량 1.80배",
            "seed_urls": [],
        },
        "domains": ["데이터센터"],
        "as_of_date": "2026-09-22",
        "language": "ko",
        "max_search_rounds": 2,  # 최초 검색 포함. 근거가 부족하면 질의를 바꿔 1회 더
        "max_revision_rounds": 1,
    },
    # 선행 기술 조사 에이전트 산출물 모사(팀원 담당)
    "technical_findings": {
        "claims": [
            {
                "claim_id": "technical:claim:001", "technology_ids": ["sw"], "topic": "원리",
                "statement": "MLA는 KV를 저차원 잠재 벡터로 압축해 캐시를 93.3% 줄인다고 보고됨",
                "basis": "direct_evidence", "evidence_ids": [],
                "conditions": ["236B MoE 기준"], "uncertainty": "자체 보고 수치",
            },
            {
                "claim_id": "technical:claim:002", "technology_ids": ["hw"], "topic": "원리",
                "statement": "ITME는 CXL-Hybrid 계층 메모리로 처리량 1.80배를 보고",
                "basis": "direct_evidence", "evidence_ids": [],
                "conditions": ["논문 실험 구성"], "uncertainty": "운영 환경 검증 불명",
            },
        ],
        "trl_estimates": [
            {"technology_id": "sw", "level": 7, "rationale": "상용 적용",
             "evidence_ids": [], "caveat": "공개 정보 기반 추정"},
            {"technology_id": "hw", "level": 4, "rationale": "연구 시연",
             "evidence_ids": [], "caveat": "공개 정보 기반 추정"},
        ],
    },
    # 관점 격리 확인용 미끼. 결과에 이 문장이 나타나면 격리가 깨진 것이다.
    "market_findings": {"claims": [{"statement": "시장 규모가 급성장 중이다"}]},
    "stakeholder_findings": {"claims": [{"statement": "업계가 CXL을 반긴다"}]},
}

from agents.domain.node import project_input

print("서브그래프에 실제로 넘어가는 입력:")
for key, value in project_input(state).items():
    print(f"  {key}: {str(value)[:80]}")

서브그래프에 실제로 넘어가는 입력:
  sw_name: DeepSeek-V2 MLA (Multi-head Latent Attention)
  hw_name: ITME (Inference Tiered Memory Expansion, CXL-Hybrid)
  as_of_date: 2026-09-22
  max_search_rounds: 2
  technical_summary: - (sw) 원리: MLA는 KV를 저차원 잠재 벡터로 압축해 캐시를 93.3% 줄인다고 보고됨
- (hw) 원리: ITME는 CXL-Hybri


## 3. 실행

LLM(GPT)과 검색 제공자는 State 밖의 런타임 의존성으로 주입합니다.
State 에는 JSON 직렬화 가능한 값만 둡니다(체크포인터 저장 경계).

웹 검색 → 본문 수집 → 긴 문서 색인이 포함되어 몇 분 걸립니다.

In [5]:
from langchain.chat_models import init_chat_model

from agents.domain import DomainAgentDeps, make_node
from agents.domain.prompts import PROMPT_VERSION
from agents.domain.runtime.manifest import LLMRunInfo, RunManifest, current_git_commit, file_checksum

MODEL = "gpt-4o-mini"

# 실행 매니페스트: 이 결과가 어떤 조건에서 나왔는지 남겨야 보고서 수치를 재현할 수 있다.
manifest = RunManifest(run_id="domain-nb-001", git_commit=current_git_commit(ROOT))
manifest.record_llm(LLMRunInfo(
    provider="openai", model=MODEL, temperature=0.0, max_tokens=None, seed=None,
    prompt_version=PROMPT_VERSION,
))

deps = DomainAgentDeps(
    llm=init_chat_model(MODEL, model_provider="openai", temperature=0),
    search_provider=search_provider,
    embedding_model="BAAI/bge-m3",  # 긴 문서 색인에만 쓰인다
    fetch_cache_dir=ROOT / "data/fetch_cache",
)

out = make_node(deps)(state)
manifest.record_embedding(deps.embedding_run_info)

# 부모 State 에 돌려주는 키. 관점별로 분리돼 병렬 분기에서 충돌하지 않는다.
print("반환 키:", sorted(out.keys()))
findings = out["domain_findings"]

/Users/sanlee/Desktop/SKALA/코딩파일/Ai-service/Capstone_Ai_RAG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 34562.13it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 54126.30it/s]

반환 키: ['domain_findings', 'evidence_store', 'quality_by_perspective', 'search_log_by_perspective']


In [6]:
# completion: 어디까지 해냈는지. pages_used 로 200페이지 한도를 확인한다.
c = findings["completion"]
print(f"status={c['status']}  검색 라운드={c['search_rounds_used']}  수집={c['pages_used']}/200p")
print(f"근거 {len(out['evidence_store'])}건 / 주장 {len(findings['claims'])}건 / 판정 {len(findings['fits'])}건")

if c["gaps"]:
    print("\n못 채운 부분(gaps) — 보고서 한계점 장으로 이어진다")
    for g in c["gaps"][:8]:
        print(" -", g)
if c["errors"]:
    print("\n오류(숨기지 않고 남긴다)")
    for e in c["errors"][:5]:
        print(" -", e)

status=partial  검색 라운드=2  수집=194/200p
근거 32건 / 주장 11건 / 판정 2건

못 채운 부분(gaps) — 보고서 한계점 장으로 이어진다
 - 전력·발열
 - 정확도 유지

오류(숨기지 않고 남긴다)
 - 본문 수집 실패(https://openreview.net/pdf?id=JHvS2Q9RtW): HTTPStatusError: Client error '403 Forbidden' for url 'https://openreview.net/pdf?id=JHvS2Q9RtW'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403
 - 본문 수집 실패(https://www.deepseek.com/news/context-caching): ConnectError: [Errno 54] Connection reset by peer
 - 본문 수집 실패(https://www.sciencedirect.com/science/article/abs/pii/S02552): HTTPStatusError: Client error '403 Forbidden' for url 'https://www.sciencedirect.com/science/article/abs/pii/S025527010600119X'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403
 - 본문 수집 실패(https://www.sciencedirect.com/science/article/pii/S025527010): HTTPStatusError: Client error '403 Forbidden' for url 'https://www.sciencedirect.com/science/article/pii/S025527010600119X'
For more information check:

## 4. 품질 검사 두 층

코드로 답할 수 있는 것을 LLM 에게 맡기면 같은 입력에 다른 판정이 나옵니다.
그래서 결정적 guard 가 먼저 보고, judge 는 대조로 답할 수 없는 것만 봅니다.

- guard: citation, locator, quote, numeric unit, date, reference integrity
- judge: coverage, neutrality
- linter: 근거 없는 승자·추천·압도 표현 차단 (비교 사실 자체는 허용)

In [7]:
q = out["quality_by_perspective"]["domain"]

print(f"guard  통과={q['guard']['passed']}  검사 대상: 주장 {q['guard']['checked_claims']}건 / 근거 {q['guard']['checked_evidence']}건")
print(f"       위반 분포: {q['guard']['counts_by_check']}")
for v in q["guard"]["violations"][:5]:
    print(f"       - [{v['check']}] {v['target_id']}: {v['detail'][:80]}")

print(f"\nlint   차단 {q['lint']['blocking_count']}건 / 경고 {q['lint']['warning_count']}건")
for f in q["lint"]["findings"][:4]:
    print(f"       - [{f['kind']}] {f['target_id']}: '{f['matched']}' ({f['detail']})")

print(f"\njudge  neutrality={q['judge']['neutrality']}  통과={q['judge']['passed']}")
print(f"       다뤄진 축 {len(q['judge']['covered_axes'])}개 / 빈 축 {q['judge']['missing_axes']}")
for note in q["judge"]["neutrality_notes"][:3]:
    print(f"       - {note}")

guard  통과=True  검사 대상: 주장 11건 / 근거 32건
       위반 분포: {}

lint   차단 0건 / 경고 0건

judge  neutrality=neutral  통과=True
       다뤄진 축 6개 / 빈 축 []


In [8]:
# fits: 종합 에이전트가 상충을 기계적으로 찾도록 열거형으로 고정한 판정
for f in findings["fits"]:
    print(f"[{f['technology_id']}] {f['domain']} 적합성: {f['assessment']}")
    print(f"  근거 주장: {f['claim_ids']}")
    for lim in f["limitations"][:3]:
        print(f"  제약: {lim}")
    print()

[sw] 데이터센터 적합성: suitable
  근거 주장: ['domain:claim:001', 'domain:claim:002', 'domain:claim:003', 'domain:claim:004', 'domain:claim:005', 'domain:claim:006']

[hw] 데이터센터 적합성: conditional
  근거 주장: ['domain:claim:007', 'domain:claim:008', 'domain:claim:009', 'domain:claim:010', 'domain:claim:011']
  제약: 신규 하드웨어가 필요하여 도입 비용이 증가할 수 있다.



In [9]:
# claims: basis 로 사실과 추론을 분리한다.
# direct_evidence = 근거가 직접 뒷받침 / inference = 유추 / unknown = 판단 보류
for cl in findings["claims"]:
    print(f"[{cl['claim_id']}] ({cl['basis']}) {'/'.join(cl['technology_ids'])}")
    print(f"  {cl['statement']}")
    print(f"  근거: {cl['evidence_ids']}")
    if cl["conditions"]:
        print(f"  전제: {cl['conditions'][:2]}")
    print(f"  한계: {cl['uncertainty'][:80]}\n")

[domain:claim:001] (direct_evidence) sw
  DeepSeek-V2는 KV 캐시를 93.3% 줄여 HBM 용량 압박을 완화할 수 있다.
  근거: ['domain:ev:e385d066b294']
  전제: ['멀티테넌시 환경에서 요청당 KV 캐시 점유가 동시 처리 요청 수를 제한하는 경우']
  한계: 

[domain:claim:002] (direct_evidence) sw
  DeepSeek-V2는 200개의 동시 요청에서 7,266 tokens/s의 처리량을 달성할 수 있다.
  근거: ['domain:ev:63eaaae4a44e']
  전제: ['멀티테넌시 환경에서 GPU 1대가 감당하는 요청 수가 단가를 결정하는 경우']
  한계: 

[domain:claim:003] (inference) sw
  DeepSeek-V2는 KV 캐시를 줄여 서비스 지연을 감소시킬 수 있다.
  근거: ['domain:ev:e385d066b294']
  전제: ['서비스 SLA를 만족해야 하며 메모리 계층 이동이 전송 지연을 유발하는 경우']
  한계: DeepSeek-V2의 지연시간에 대한 구체적인 수치가 부족하다.

[domain:claim:004] (direct_evidence) sw
  DeepSeek-V2는 KV 캐시를 줄이면서도 모델 품질을 유지할 수 있다.
  근거: ['domain:ev:e385d066b294']
  전제: ['압축률을 높이면 품질이 떨어질 수 있는 경우']
  한계: 

[domain:claim:005] (inference) sw
  DeepSeek-V2는 메모리 사용량을 줄여 전력 소모를 감소시킬 수 있다.
  근거: ['domain:ev:e385d066b294']
  전제: ['랙 단위 전력 예산이 상한인 경우']
  한계: 전력 소모에 대한 구체적인 수치가 부족하다.

[domain:claim:006] (direct_evidence) sw
  DeepSeek-V2는 기존 서버에 즉시 적용 가능하여 도입 비

In [10]:
# evidence_store: 보고서 REFERENCE 장을 만들 수 있는 형태인지 확인한다.
# locator 와 quote 가 있어야 guard 의 원문 대조와 인용 추적이 가능하다.
from collections import Counter

print("출처 등급 분포:", Counter(e["source_type"] for e in out["evidence_store"].values()))
for ev in list(out["evidence_store"].values())[:3]:
    print(f"\n[{ev['evidence_id']}] ({ev['source_type']}) {ev['locator']}")
    print(f"  {ev['title'][:70]}")
    print(f"  {ev['url'][:80]}")
    print(f"  인용: {ev['quote'][:110]}...")

출처 등급 분포: Counter({'paper': 28, 'vendor': 4})

[domain:ev:26b04a964dd5] (vendor) chars:0-3000+0
  DeepSeek API introduces Context Caching on Disk, cutting prices by an 
  https://api-docs.deepseek.com/news/news0802
  인용: DeepSeek API introduces Context Caching on Disk, cutting prices by an order of magnitude In large language mod...

[domain:ev:da13b0180fbb] (vendor) chars:0-1158
  DeepSeek API 创新采用硬盘缓存，价格再降一个数量级 | DeepSeek API Docs
  https://api-docs.deepseek.com/zh-cn/news/news0802
  인용: DeepSeek API 创新采用硬盘缓存，价格再降一个数量级 在大模型 API 的使用场景中，用户的输入有相当比例是重复的。举例说，用户的 prompt 往往有一些重复引 用的部分；再举例说，多轮对话中，每一轮都要将前...

[domain:ev:74b7aa45b204] (paper) chars:30000-33000+1000
  DeepSeek-V2: A Strong, Economical, and Efficient Mixture-of-Experts La
  https://arxiv.org/html/2405.04434v2
  인용: can attain a relatively high Model FLOPs Utilization (MFU). During our practical training on the H800 cluster,...


In [11]:
# 검색 로그: 어떤 질의가 어떤 출처를 데려왔고 무엇을 왜 걸렀는지 남는다.
log = out["search_log_by_perspective"]["domain"]
accepted = sum(len(entry["accepted_urls"]) for entry in log)
rejected = sum(len(entry["rejected"]) for entry in log)
print(f"질의 {len(log)}건 / 채택 {accepted} / 거부 {rejected}")

for entry in log[:3]:
    print(f"\nQ: {entry['query'][:70]}  (캐시={entry['from_cache']})")
    for url in entry["accepted_urls"][:2]:
        print(f"   채택 {url[:70]}")
    for item in entry["rejected"][:2]:
        print(f"   거부 {item['url'][:60]} <- {item['reason']}")

질의 12건 / 채택 67 / 거부 5

Q: What is the maximum number of concurrent requests that can be handled   (캐시=True)
   채택 https://openreview.net/pdf?id=JHvS2Q9RtW
   채택 https://api-docs.deepseek.com/news/news0802

Q: What is the minimum number of concurrent requests that lead to perform  (캐시=True)
   채택 https://arxiv.org/html/2603.10031v1
   채택 https://arxiv.org/html/2502.07864v4

Q: How does ITME's CXL-Hybrid memory architecture impact the throughput (  (캐시=True)
   채택 https://arxiv.org/html/2606.12556v1
   채택 https://arxiv.org/html/2606.12556v2


In [12]:
# 관점 격리 검증: 미끼로 넣은 시장·이해관계자 문장이 결과에 새지 않았는지 확인한다.
blob = json.dumps(out, ensure_ascii=False)
leaks = [s for s in ("시장 규모가 급성장", "업계가 CXL을 반긴다") if s in blob]
print("관점 격리:", "누수 없음" if not leaks else f"누수 {leaks}")

# State 저장 경계: JSON 직렬화 가능해야 체크포인터에 들어간다.
print(f"직렬화 크기: {len(blob):,} bytes")

관점 격리: 누수 없음
직렬화 크기: 49,457 bytes


## 5. 재현성 기록

보고서에 인용한 수치를 나중에 되짚으려면 그 결과가 어떤 조건에서 나왔는지 남아야 합니다.
모델 버전이나 검색 캐시가 바뀌면 같은 질문에도 다른 답이 나오기 때문입니다.

In [13]:
out_dir = ROOT / "outputs"
out_dir.mkdir(exist_ok=True)
result_path = out_dir / "domain_findings_demo.json"
result_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")

manifest.record_artifact(file_checksum(result_path))
manifest.record_artifact(file_checksum(out_dir / "ablation.json"))
manifest_path = manifest.finish().save(out_dir / "run_manifest.json")

print("LLM      :", manifest.llm)
print("임베딩   :", manifest.embedding)
print("하드웨어 :", {k: manifest.hardware.get(k) for k in ("os", "machine", "ram_gb", "torch_backend")})
print("git      :", (manifest.git_commit or "")[:12])
print("\n산출물 체크섬")
for a in manifest.artifacts:
    print(f"  {a['name']:<28} {a['sha256'][:16]}  {a['bytes']:,}B")
print(f"\n저장: {manifest_path}")

LLM      : {'provider': 'openai', 'model': 'gpt-4o-mini', 'temperature': 0.0, 'max_tokens': None, 'seed': None, 'prompt_version': 'domain/v2-websearch'}
임베딩   : {'model_name': 'BAAI/bge-m3', 'device': 'mps:0', 'normalize_embeddings': True, 'batch_size': 16, 'dimension': 1024, 'torch_version': '2.14.0', 'platform': 'Darwin arm64'}
하드웨어 : {'os': 'Darwin 25.5.0', 'machine': 'arm64', 'ram_gb': 16.0, 'torch_backend': 'mps'}
git      : 260dcf06979c

산출물 체크섬
  domain_findings_demo.json    bee4e12dfb597458  59,870B
  ablation.json                93a11510c2db6e75  2,036B

저장: /Users/sanlee/Desktop/SKALA/코딩파일/Ai-service/Capstone_Ai_RAG/outputs/run_manifest.json


## 6. 검색 방식 비교 결과

`python -m agents.domain.evaluation.ablation --embedding BAAI/bge-m3` 로 재현합니다.
Golden Set 은 정답을 청크 ID 가 아니라 "반드시 들어 있어야 할 표지 문자열"로 정의했습니다.
청킹 파라미터를 바꿔도 같은 기준으로 비교하기 위해서입니다.

In [14]:
ablation_path = ROOT / "outputs/ablation.json"
if ablation_path.exists():
    report = json.loads(ablation_path.read_text(encoding="utf-8"))
    print(f"청크 {report['chunk_count']}개 / 질의 {report['golden_set_size']}개")
    header = f"{'method':<26}{'Hit@1':>7}{'Hit@3':>7}{'Hit@5':>7}{'MRR':>7}{'mean ms':>9}{'peak MB':>9}"
    print(header)
    print("-" * len(header))
    for row in report["results"]:
        print(f"{row['method']:<26}{row['hit_at_1']:>7}{row['hit_at_3']:>7}"
              f"{row['hit_at_5']:>7}{row['mrr']:>7}{row['mean_latency_ms']:>9}{row['peak_memory_mb']:>9}")
else:
    print("ablation.json 없음 — python -m agents.domain.evaluation.ablation 을 먼저 실행하세요")

청크 213개 / 질의 16개
method                      Hit@1  Hit@3  Hit@5    MRR  mean ms  peak MB
------------------------------------------------------------------------
bm25                        0.562  0.625  0.688  0.606      0.2      0.0
dense:BAAI/bge-m3           0.875  0.875  0.875  0.875     35.3      0.1
hybrid:bm25+dense           0.812  0.875  0.875  0.833     33.7      0.1


## 정리와 남은 위험

- 검색 대상은 공신력 등급표에 있는 웹 출처뿐이며, 제공자 필터를 통과한 결과도 다시 거릅니다.
- 임베딩은 긴 문서 색인에만 쓰입니다. 짧은 문서는 색인 없이 그대로 근거가 됩니다.
- 근거가 부족하면 메우지 않고 `unknown` 과 `gaps` 로 남깁니다.

남은 위험:
- hybrid 융합 가중치를 조정하지 않았습니다. 측정에서는 dense 단독이 MRR 기준 앞섭니다.
- Golden Set 이 16문항이라 한 문항이 지표의 0.06을 좌우합니다.
- 200페이지 한도는 런타임 누적으로 막습니다. 어떤 페이지가 들어갔는지는 실행 후
  `search_log` 와 `evidence_store` 로만 확인됩니다.